In [1]:
!pip -q install imagecorruptions
import time, numpy as np, pandas as pd
from PIL import Image
import skimage.filters as skf
import imagecorruptions.corruptions as ic
from imagecorruptions import corrupt, get_corruption_names

np.float_ = np.float64                      # fix for fog (NumPy 2)
_g = skf.gaussian
def gaussian(x, sigma, multichannel=None, **kw):   # fix for glass_blur (new scikit-image)
    if multichannel: kw['channel_axis'] = -1
    return _g(x, sigma=sigma, **kw)
ic.gaussian = gaussian

NB = '/kaggle/input/notebooks/aadiparkhi'
pl = pd.read_csv(f'{NB}/01a-data-curation/places_manifest.csv')
bd = pd.read_csv(f'{NB}/01a2-bdd-curation/bdd_manifest.csv')
ims = {'places': np.array(Image.open(f"{NB}/01a-data-curation/data/{pl[pl.split == 'test'].path.iloc[0]}")),
       'bdd': np.array(Image.open(f"{NB}/01a2-bdd-curation/data/{bd[bd.split == 'test'].path.iloc[0]}"))}
names = get_corruption_names()
for d, im in ims.items():
    for c in names:
        t = time.time(); out = corrupt(im, corruption_name=c, severity=3)
        print(d, c, out.shape, round(time.time() - t, 3))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 7.1 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/imagecorruptions/corruptions.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


places gaussian_noise (224, 224, 3) 0.01
places shot_noise (224, 224, 3) 0.014
places impulse_noise (224, 224, 3) 0.004
places defocus_blur (224, 224, 3) 0.07
places glass_blur (224, 224, 3) 1.212
places motion_blur (224, 224, 3) 0.011
places zoom_blur (224, 224, 3) 0.072
places snow (224, 224, 3) 0.017
places frost (224, 224, 3) 0.049
places fog (224, 224, 3) 0.004
places brightness (224, 224, 3) 0.047
places contrast (224, 224, 3) 0.003
places elastic_transform (224, 224, 3) 0.02
places pixelate (224, 224, 3) 0.001
places jpeg_compression (224, 224, 3) 0.027
bdd gaussian_noise (216, 384, 3) 0.006
bdd shot_noise (216, 384, 3) 0.016
bdd impulse_noise (216, 384, 3) 0.004
bdd defocus_blur (216, 384, 3) 0.01
bdd glass_blur (216, 384, 3) 2.013
bdd motion_blur (216, 384, 3) 0.012
bdd zoom_blur (216, 384, 3) 0.12
bdd snow (216, 384, 3) 0.015
bdd frost (216, 384, 3) 0.017
bdd fog (216, 384, 3) 0.007
bdd brightness (216, 384, 3) 0.028
bdd contrast (216, 384, 3) 0.004
bdd elastic_transform (216

In [2]:
rs = 42
bp = pl[pl.split == 'test'].groupby('label').sample(8, random_state=rs)
bb = pd.concat([g.sample(min(len(g), 38), random_state=rs) for _, g in bd[bd.split == 'test'].groupby('cond')])
bp.to_csv('/kaggle/working/base_places.csv', index=False)
bb.to_csv('/kaggle/working/base_bdd.csv', index=False)
print(len(bp), len(bb))
print(bb.cond.value_counts())
print('person rate', bb.person.mean())

384 404
cond
clear_day            38
clear_night          38
clear_twilight       38
overcast_day         38
overcast_twilight    38
rainy_day            38
rainy_night          38
snow_fog_day         38
snow_fog_night       38
snow_fog_twilight    35
rainy_twilight       27
Name: count, dtype: int64
person rate 0.20297029702970298


In [3]:
import os, multiprocessing as mp
OUT = '/kaggle/working/corrupt'
CLEAN = {'places': f'{NB}/01a-data-curation/data', 'bdd': f'{NB}/01a2-bdd-curation/data'}

def work(a):
    dom, i, rel = a
    np.random.seed(i + (0 if dom == 'places' else 100000))
    im = np.array(Image.open(f'{CLEAN[dom]}/{rel}').convert('RGB'))
    rec = []
    for c in ['clean'] + names:
        for s in ([0] if c == 'clean' else range(1, 6)):
            out = im if c == 'clean' else corrupt(im, corruption_name=c, severity=s)
            p = f'{dom}/{c}_{s}/{i:04d}.jpg'
            os.makedirs(os.path.dirname(f'{OUT}/{p}'), exist_ok=True)
            Image.fromarray(out).save(f'{OUT}/{p}', quality=95)
            rec.append((dom, i, c, s, p))
    return rec

def run(dom, df, n=None):
    t = time.time()
    tasks = [(dom, i, r) for i, r in enumerate(df.path)][:n]
    with mp.get_context('fork').Pool(4) as pool:
        recs = [r for out in pool.imap_unordered(work, tasks) for r in out]
    m = pd.DataFrame(recs, columns=['domain', 'bid', 'corruption', 'severity', 'path'])
    print(dom, len(m), round(time.time() - t), 's')
    return m

m1 = run('places', bp, 8); m2 = run('bdd', bb, 8)

places 608 19 s
bdd 608 32 s


In [4]:
mp_ = run('places', bp)
mb_ = run('bdd', bb)
mp_.to_csv('/kaggle/working/corrupt_places_manifest.csv', index=False)
mb_.to_csv('/kaggle/working/corrupt_bdd_manifest.csv', index=False)
!du -sh /kaggle/working/corrupt

places 29184 899 s
bdd 30704 1569 s
1.8G	/kaggle/working/corrupt
